In [6]:
from pyspark.sql.functions import (
    col,
    concat,
    year,
    month,
    dayofmonth,
    quarter,
    date_format,
    lpad,
    to_date
)

# following import is for saving data frame into warehouse table i.e. synapsesql
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

# for calling the stored procedure
import pyodbc
import struct
from notebookutils import mssparkutils

StatementMeta(, d0eb3965-fe09-47b8-940d-87a8df5836dc, 10, Finished, Available, Finished, False)

In [7]:
# variables

WAREHOUSE = "WH_GTR"

DIM_DATE = f"{WAREHOUSE}.dbo.DimDate"

GOLD_REFRESH_QUEUE = "LH_Gold.dbo.gold_refresh_queue"
SILVER_GTR_PASSENGER_LOADING = "LH_Silver.dbo.silver_gtr_passenger_loading"

WAREHOUSE_SERVER = "ifszlywihgmehbiffisddvsmxq-36zqqynnw5eufg3h2ddumt7usi.datawarehouse.fabric.microsoft.com"


StatementMeta(, d0eb3965-fe09-47b8-940d-87a8df5836dc, 11, Finished, Available, Finished, False)

# Load common functions

In [8]:
%run NB_03_00_Gold_Common

StatementMeta(, d0eb3965-fe09-47b8-940d-87a8df5836dc, 15, Finished, Available, Finished, True)

In [9]:
print("Getting gold refresh queue data.")
gold_refresh_queue_df = spark.read.table(GOLD_REFRESH_QUEUE).filter(col("status") == "PENDING")
#display(gold_refresh_queue_df)

print("Getting pending period.")
pending_periods = get_pending_periods(
    gold_refresh_queue_df
)

print("Getting silver data.")
silver_filtered_df = (
    spark.read.table(SILVER_GTR_PASSENGER_LOADING)
    .filter(col("rail_period").isin(pending_periods))
)

print("Getting silver filter data.")
dim_date_df = build_dim_date(
    silver_filtered_df
)

print (pending_periods)
print("Delete existing DimDate data.")
delete_data_by_rail_period(
    pending_periods,
    WAREHOUSE_SERVER,
    WAREHOUSE
)

print("Load DimDate new data.")
load_table(
    dim_date_df,
    DIM_DATE
)

StatementMeta(, d0eb3965-fe09-47b8-940d-87a8df5836dc, 16, Finished, Available, Finished, False)

Getting gold refresh queue data.
Getting pending period.
Getting silver data.
Getting silver filter data.
['2706', '2704', '2705']
Delete existing DimDate data.
Deleting data for : 2706
Deleted DimDate rows for RailPeriod: 2706
Deleting data for : 2704
Deleted DimDate rows for RailPeriod: 2704
Deleting data for : 2705
Deleted DimDate rows for RailPeriod: 2705
Load DimDate new data.
